# ECT Supplementary Analysis
## Estimator Collapse Theory — Extended Validation & Sensitivity Notebook

**Authors:** N. Barua, R. J. Douglas  
**Affiliation:** AN Holdings CO. / Kobe Gakuin University  
**DOI:** 10.5281/zenodo.20037820  

---

### Purpose
This notebook extends the core Monte Carlo results of Section II-E with three supplementary contributions:

| Section | Contribution | Paper claim upgraded |
|---------|-------------|---------------------|
| **B** | MC propagation on the Γ_crit boundary | Point estimate → 95% CI |
| **C** | Tornado analysis on Vulnerability Index V_i | Qualitative sensitivity → quantitative ranking |
| **D** | Continuous CEP vs perturbation amplitude curves | Operating points with mission profiles |
| **E** | NIS compliance surface over (A, ω) | Scalar result → 2-D parameter landscape |
| **F** | η_info vs energy budget | Order-of-magnitude → continuous curve |

**CRediT roles engaged:** Conceptualisation · Formal Analysis · Methodology · Software · Validation · Visualisation

---

In [ ]:
import sys, os
import numpy as np
from scipy.stats import chi2
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

# ── Import core simulation module ─────────────────────────────────────────────
REPO = os.path.abspath('.')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import ECT_3D_Simulation_v141 as ECT

# ── Supplementary run config ─────────────────────────────────────────────────
# Use N=100 for interactive exploration; set to 500 for publication runs
N_SUPP   = 100
N_SWEEP  = 40    # runs per parameter point in sweeps

print(f'ECT module loaded from: {ECT.__file__}')
print(f'N_SUPP={N_SUPP}  N_SWEEP={N_SWEEP}  T={ECT.T}s  Δt={ECT.DT}s')

---
## Section A — Baseline MC Validation
Reproduce core paper metrics at N=N_SUPP. Confirm Γ(t), CEP, and NIS results.

In [ ]:
print('Running baseline MC ...')
res = ECT.run_mc(n_mc=N_SUPP, verbose=True)
ECT.print_summary(res)

In [ ]:
gamma_t, gamma_all = ECT.gamma_series(res)
t = res['t']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Γ(t) ─────────────────────────────────────────────────────────────────────
ax = axes[0]
p5, p95 = np.percentile(gamma_all, [5, 95], axis=0)
ax.fill_between(t, p5, p95, alpha=0.15, color='#1f77b4')
ax.plot(t, gamma_t, color='#1f77b4', lw=1.7, label='Mean Γ(t) — perturbed')
ax.axhline(ECT.GAMMA_CRIT, color='#d62728', ls='--', lw=1.4, label=f'Γ_crit={ECT.GAMMA_CRIT}')
ax.axhline(1.0, color='#2ca02c', ls='--', lw=1.4, label='Nominal')
ax.set(xlabel='Time [s]', ylabel='Γ(t)', title='Estimator Instability', xlim=(0, ECT.T))
ax.legend(fontsize=8)

# ── CEP ──────────────────────────────────────────────────────────────────────
ax = axes[1]
nom_med  = np.median(res['nom_cep'], axis=0)
pert_med = np.median(res['pert_cep'], axis=0)
p5c, p95c = np.percentile(res['pert_cep'], [5, 95], axis=0)
ax.fill_between(t, p5c, p95c, alpha=0.15, color='#ff7f0e')
ax.plot(t, pert_med, color='#ff7f0e', lw=1.7, label='Perturbed CEP')
ax.plot(t, nom_med, color='#2ca02c', lw=1.7, ls='--', label='Nominal CEP')
ax.axhline(ECT.R_L, color='#d62728', ls=':', lw=1.4, label=f'R_L={ECT.R_L}m')
ax.set(xlabel='Time [s]', ylabel='CEP [m]', title='Circular Error Probable', xlim=(0, ECT.T))
ax.legend(fontsize=8)

# ── NIS ──────────────────────────────────────────────────────────────────────
ax = axes[2]
nom_nis  = np.median(res['nom_nis'], axis=0)
pert_nis = np.median(res['pert_nis'], axis=0)
ax.plot(t, pert_nis, color='#1f77b4', lw=1.7, alpha=0.9, label='Perturbed NIS')
ax.plot(t, nom_nis, color='#2ca02c', lw=1.7, ls='--', label='Nominal NIS')
ax.axhline(ECT.CHI2_GATE, color='#d62728', ls=':', lw=1.6, label=f'Gate χ²₃={ECT.CHI2_GATE:.2f}')
ax.set(xlabel='Time [s]', ylabel='NIS', title='Innovation Gate Compliance', xlim=(0, ECT.T))
ax.legend(fontsize=8)

fig.suptitle('Section A — Baseline MC Validation', fontsize=11, fontweight='bold')
fig.tight_layout()
plt.show()

nom_ss, pert_ss = ECT.cep_steady(res)
nc, pc = ECT.nis_compliance(res)
frac = (gamma_all[:,-1] >= ECT.GAMMA_CRIT).mean()*100
print(f'\nKey metrics (N={N_SUPP}):')
print(f'  Γ(t) > Γ_crit in {frac:.0f}% of runs')
print(f'  Nom CEP={nom_ss:.2f}m  Pert CEP={pert_ss:.2f}m  Δ={100*(pert_ss/nom_ss-1):.0f}%')
print(f'  NIS compliance — nom={nc*100:.1f}%  pert={pc*100:.1f}%')
print(f'  MKI = {pert_ss/ECT.R_L:.3f}  (R_L={ECT.R_L}m)')

---
## Section B — Γ_crit Boundary: Point Estimate → 95% Confidence Interval

**Paper (Section II-E):** Reports Γ_crit = 6.5 as a fixed design threshold derived from
eq. (7): Γ_crit ≈ (CEP_MK / CEP_nom)².

**This section:** Propagates uncertainty in the system parameters (Q, R_GNSS, A_pert)
through the MC to produce a **95% CI on the Γ_crit crossing time** and the
**fraction of runs exceeding Γ_crit** under realistic parameter uncertainty.

### Method
Treat key parameters as uncertain:
- Q_pos: ± 50% of nominal (process noise — platform model uncertainty)
- R_GNSS: ± 30% (sensor noise estimation error)
- A_pert: ± 25% (perturbation amplitude uncertainty)

Latin Hypercube sample N_LHC parameter sets → run short MC for each → aggregate
the distribution of (Γ_final, crossing_time, NIS_compliance).

In [ ]:
from scipy.stats import qmc

N_LHC   = 40      # parameter ensemble size
N_MC_B  = 30      # MC runs per parameter set

# Parameter bounds (relative to nominal)
# [Q_scale, R_scale, A_scale]
sampler = qmc.LatinHypercube(d=3, seed=42)
lhc_raw = sampler.random(n=N_LHC)
lhc_scaled = qmc.scale(lhc_raw,
    l_bounds=[0.5, 0.7, 0.75],
    u_bounds=[1.5, 1.3, 1.25]
)

gamma_finals   = np.empty((N_LHC, N_MC_B))
nis_compliance = np.empty((N_LHC,))
cep_perturbed  = np.empty((N_LHC,))

print(f'Running {N_LHC} parameter sets × {N_MC_B} MC runs ...')

for idx, (q_s, r_s, a_s) in enumerate(tqdm(lhc_scaled)):
    # Temporarily patch ECT module parameters
    orig_Q      = ECT.Q.copy()
    orig_R      = ECT.R_GNSS.copy()
    orig_A      = ECT.A_PERT

    ECT.Q       = orig_Q * q_s
    ECT.R_GNSS  = orig_R * r_s
    ECT.A_PERT  = orig_A * a_s

    r_tmp = ECT.run_mc(n_mc=N_MC_B, verbose=False)

    _, ga = ECT.gamma_series(r_tmp)
    gamma_finals[idx] = ga[:, -1]
    nc_i, pc_i = ECT.nis_compliance(r_tmp)
    nis_compliance[idx] = pc_i
    _, pert_ss = ECT.cep_steady(r_tmp)
    cep_perturbed[idx] = pert_ss

    # Restore
    ECT.Q, ECT.R_GNSS, ECT.A_PERT = orig_Q, orig_R, orig_A

# Fraction of runs exceeding Γ_crit, per parameter set
frac_exceed = (gamma_finals >= ECT.GAMMA_CRIT).mean(axis=1)  # (N_LHC,)

print(f'\nFraction exceeding Γ_crit={ECT.GAMMA_CRIT}:')
print(f'  Mean={frac_exceed.mean()*100:.1f}%  '
      f'95% CI=[{np.percentile(frac_exceed,2.5)*100:.1f}%, {np.percentile(frac_exceed,97.5)*100:.1f}%]')
print(f'NIS compliance: {nis_compliance.mean()*100:.1f}% '
      f'CI=[{np.percentile(nis_compliance,2.5)*100:.1f}%, {np.percentile(nis_compliance,97.5)*100:.1f}%]')
print(f'Perturbed CEP: {cep_perturbed.mean():.2f}m '
      f'CI=[{np.percentile(cep_perturbed,2.5):.2f}m, {np.percentile(cep_perturbed,97.5):.2f}m]')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Frac exceed histogram ─────────────────────────────────────────────────────
ax = axes[0]
ax.hist(frac_exceed * 100, bins=15, color='#1f77b4', alpha=0.75, edgecolor='white')
lo, hi = np.percentile(frac_exceed*100, [2.5, 97.5])
ax.axvline(lo, color='#d62728', ls='--', lw=1.4, label=f'95% CI [{lo:.0f}%, {hi:.0f}%]')
ax.axvline(hi, color='#d62728', ls='--', lw=1.4)
ax.axvline(frac_exceed.mean()*100, color='k', lw=1.6, label=f'Mean {frac_exceed.mean()*100:.0f}%')
ax.set(xlabel='Runs exceeding Γ_crit [%]', ylabel='Parameter sets', title='Γ_crit Exceedance Distribution')
ax.legend(fontsize=8)

# ── CEP distribution ─────────────────────────────────────────────────────────
ax = axes[1]
ax.hist(cep_perturbed, bins=15, color='#ff7f0e', alpha=0.75, edgecolor='white')
c_lo, c_hi = np.percentile(cep_perturbed, [2.5, 97.5])
ax.axvline(c_lo, color='#d62728', ls='--', lw=1.4, label=f'95% CI [{c_lo:.1f}, {c_hi:.1f}] m')
ax.axvline(c_hi, color='#d62728', ls='--', lw=1.4)
ax.axvline(ECT.R_L, color='k', ls=':', lw=1.4, label=f'R_L={ECT.R_L}m')
ax.set(xlabel='Steady-state CEP [m]', ylabel='Parameter sets', title='CEP Under Parameter Uncertainty')
ax.legend(fontsize=8)

# ── NIS compliance ────────────────────────────────────────────────────────────
ax = axes[2]
ax.hist(nis_compliance * 100, bins=12, color='#9467bd', alpha=0.75, edgecolor='white')
n_lo, n_hi = np.percentile(nis_compliance*100, [2.5, 97.5])
ax.axvline(n_lo, color='#d62728', ls='--', lw=1.4, label=f'95% CI [{n_lo:.0f}%, {n_hi:.0f}%]')
ax.axvline(n_hi, color='#d62728', ls='--', lw=1.4)
ax.set(xlabel='NIS gate compliance [%]', ylabel='Parameter sets', title='NIS Compliance Under Uncertainty')
ax.legend(fontsize=8)

fig.suptitle('Section B — Γ_crit Boundary: 95% CI via Latin Hypercube MC Propagation', fontsize=11, fontweight='bold')
fig.tight_layout()
plt.show()

print('\nSummary Table (Section B):')
print(f'{"Metric":<30} {"Mean":>10} {"95% CI Low":>12} {"95% CI High":>13}')
print('-'*70)
print(f'{"% runs > Γ_crit":<30} {frac_exceed.mean()*100:>10.1f} {lo:>12.1f} {hi:>13.1f}')
print(f'{"Perturbed CEP [m]":<30} {cep_perturbed.mean():>10.2f} {c_lo:>12.2f} {c_hi:>13.2f}')
print(f'{"NIS compliance [%]":<30} {nis_compliance.mean()*100:>10.1f} {n_lo:>12.1f} {n_hi:>13.1f}')

---
## Section C — Vulnerability Index V_i : Tornado Sensitivity Analysis

**Paper (Section IV-A, eq. 13):**
$$V_i = \frac{N_{\text{sensors}} \times f_{\text{update}}}{R_{\text{hardening}}}$$

The paper provides indicative V_i values for four system classes but does not
quantify which component drives sensitivity most.

**This section:** One-at-a-time (OAT) sensitivity analysis on V_i. Each component
varied ±50% from the High-Precision baseline (N_s=4, f=20 Hz, R_h=1.2 → V_i=66.7).
The resulting tornado chart ranks components by absolute V_i impact — providing
the quantitative sensitivity claim that is qualitative in the paper.

**Extended:** We also sweep V_i against simulated Γ(t) to show the empirical
relationship between V_i and estimator instability.

In [ ]:
# ── Vulnerability Index definition (eq. 13) ───────────────────────────────────
def vi(n_s, f, r_h):
    return (n_s * f) / r_h

# High-precision baseline (Table I)
N_S_BASE = 4.
F_BASE   = 20.
R_H_BASE = 1.2
V_BASE   = vi(N_S_BASE, F_BASE, R_H_BASE)

# OAT sensitivity: ±50% variation
delta = 0.50

params = {
    'N_sensors':    (N_S_BASE,  N_S_BASE  * (1-delta), N_S_BASE  * (1+delta)),
    'f_update (Hz)':(F_BASE,    F_BASE    * (1-delta), F_BASE    * (1+delta)),
    'R_hardening':  (R_H_BASE,  R_H_BASE  * (1-delta), R_H_BASE  * (1+delta)),
}

tornado = []
for name, (nom, lo_val, hi_val) in params.items():
    if name == 'N_sensors':
        vi_lo = vi(lo_val, F_BASE, R_H_BASE)
        vi_hi = vi(hi_val, F_BASE, R_H_BASE)
    elif name == 'f_update (Hz)':
        vi_lo = vi(N_S_BASE, lo_val, R_H_BASE)
        vi_hi = vi(N_S_BASE, hi_val, R_H_BASE)
    else:  # R_hardening
        vi_lo = vi(N_S_BASE, F_BASE, lo_val)
        vi_hi = vi(N_S_BASE, F_BASE, hi_val)
    tornado.append((name, vi_lo - V_BASE, vi_hi - V_BASE))

# Sort by total swing (descending)
tornado.sort(key=lambda x: abs(x[2]-x[1]), reverse=True)

print(f'Baseline V_i = {V_BASE:.2f}  (N_s={N_S_BASE}, f={F_BASE}Hz, R_h={R_H_BASE})')
print(f'\nTornado table (±{delta*100:.0f}% OAT):')
print(f'{"Parameter":<20} {"V_i LOW":>10} {"V_i NOM":>10} {"V_i HIGH":>10} {"Swing":>8}')
print('-'*65)
for name, dlo, dhi in tornado:
    print(f'{name:<20} {V_BASE+dlo:>10.1f} {V_BASE:>10.1f} {V_BASE+dhi:>10.1f} {abs(dhi-dlo):>8.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Tornado chart ─────────────────────────────────────────────────────────────
ax = axes[0]
colors_lo = '#1f77b4'
colors_hi = '#ff7f0e'
y_pos = np.arange(len(tornado))
labels = [t[0] for t in tornado]

for i, (name, dlo, dhi) in enumerate(tornado):
    ax.barh(i, dlo, left=V_BASE, color=colors_lo, alpha=0.8, height=0.5)
    ax.barh(i, dhi, left=V_BASE, color=colors_hi, alpha=0.8, height=0.5)
    ax.text(V_BASE + dlo - 0.5, i, f'{V_BASE+dlo:.0f}', va='center', ha='right', fontsize=9)
    ax.text(V_BASE + dhi + 0.5, i, f'{V_BASE+dhi:.0f}', va='center', ha='left', fontsize=9)

ax.axvline(V_BASE, color='k', lw=1.4, ls='--', label=f'Baseline V_i={V_BASE:.1f}')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set(xlabel='Vulnerability Index V_i', title='Tornado: OAT Sensitivity on V_i')
ax.legend(fontsize=9)

# Blue/orange legend patches
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=colors_lo, alpha=0.8, label=f'Low ({(1-delta)*100:.0f}% of nominal)'),
    Patch(color=colors_hi, alpha=0.8, label=f'High ({(1+delta)*100:.0f}% of nominal)'),
], fontsize=9)

# ── System class V_i bar chart (Table I) ──────────────────────────────────────
ax = axes[1]
sys_classes = ['Single-Sensor\nINS', 'Dual-Modal\nMidcourse', 'High-Precision\nMulti-Sensor', 'High-Dynamics\nMulti-Sensor']
vi_values   = [vi(1,1,1.0), vi(3,10,1.0), vi(4,20,1.2), vi(4,80,1.0)]
colors_bar  = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']

bars = ax.bar(sys_classes, vi_values, color=colors_bar, alpha=0.8, edgecolor='white', width=0.6)
for bar, val in zip(bars, vi_values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 3, f'{val:.0f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set(ylabel='V_i (indicative)', title='V_i by System Class (Table I, paper)')
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())

fig.suptitle('Section C — Vulnerability Index: Tornado Analysis', fontsize=11, fontweight='bold')
fig.tight_layout()
plt.show()

print('\nKey finding: f_update and N_sensors have equal fractional impact on V_i')
print('R_hardening has INVERTED sensitivity (hardening reduces V_i nonlinearly).')

---
## Section D — CEP vs Perturbation Amplitude: Continuous Curves

**Paper (Section II-E):** Reports a single operating point: A=1.2m → CEP_pert=7.9m.

**This section:** Sweeps A_pert ∈ [0, A_max] continuously, where A_max is the maximum
amplitude that stays within the χ²₃=7.815 innovation gate with ≥ 90% compliance.

The curves show:
- CEP vs A for multiple mission tolerance profiles (R_L = 5, 10, 15, 25 m)
- The paper operating point marked
- SMK boundary for each profile
- The gate-compliance envelope (secondary axis)

In [ ]:
N_AMP_SWEEP = 20   # amplitude grid points
N_MC_D      = 30   # runs per amplitude

A_grid = np.linspace(0., 2.2, N_AMP_SWEEP)
cep_mean   = np.empty(N_AMP_SWEEP)
cep_p5     = np.empty(N_AMP_SWEEP)
cep_p95    = np.empty(N_AMP_SWEEP)
nis_comp   = np.empty(N_AMP_SWEEP)

print(f'Sweeping A_pert over {N_AMP_SWEEP} points × {N_MC_D} MC runs ...')

orig_A = ECT.A_PERT
for j, A in enumerate(tqdm(A_grid)):
    ECT.A_PERT = A
    if A == 0.:
        r_tmp = ECT.run_mc(n_mc=N_MC_D, verbose=False)
        _, ps = ECT.cep_steady(r_tmp)
        cep_mean[j] = ps
        cep_p5[j] = ps
        cep_p95[j] = ps
        nis_comp[j] = 1.0
    else:
        r_tmp = ECT.run_mc(n_mc=N_MC_D, verbose=False)
        per_run_cep = r_tmp['pert_cep'][:, -200:].mean(axis=1)
        cep_mean[j] = np.mean(per_run_cep)
        cep_p5[j]   = np.percentile(per_run_cep, 5)
        cep_p95[j]  = np.percentile(per_run_cep, 95)
        _, pc = ECT.nis_compliance(r_tmp)
        nis_comp[j] = pc

ECT.A_PERT = orig_A
print('Sweep complete.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── CEP vs amplitude ──────────────────────────────────────────────────────────
ax = axes[0]
ax.fill_between(A_grid, cep_p5, cep_p95, alpha=0.15, color='#ff7f0e')
ax.plot(A_grid, cep_mean, color='#ff7f0e', lw=2.0, label='Mean perturbed CEP')

mission_profiles = [
    (5.,   '#8B0000', '-.',  'R_L=5m  (precision)'),
    (10.,  '#d62728', '--',  'R_L=10m (terminal)'),
    (15.,  '#ff7f0e', ':',   'R_L=15m (paper baseline)'),
    (25.,  '#9467bd', '-',   'R_L=25m (midcourse)'),
]
for rl, col, ls, lbl in mission_profiles:
    ax.axhline(rl, color=col, ls=ls, lw=1.3, alpha=0.8, label=lbl)

# Paper operating point
ax.axvline(1.2, color='k', ls='--', lw=1.2, alpha=0.7)
ax.scatter([1.2], [cep_mean[np.argmin(np.abs(A_grid-1.2))]],
           s=80, color='k', zorder=5, label='Paper op. point (A=1.2m)')

ax.set(xlabel='Perturbation Amplitude A [m]', ylabel='Steady-state CEP [m]',
       title='CEP vs Perturbation Amplitude')
ax.legend(fontsize=8)

# ── NIS compliance vs amplitude ───────────────────────────────────────────────
ax = axes[1]
ax.plot(A_grid, nis_comp * 100, color='#1f77b4', lw=2.0, marker='o', ms=4)
ax.axhline(90., color='#d62728', ls='--', lw=1.4, label='90% compliance floor')
ax.axhline(95., color='#2ca02c', ls='--', lw=1.4, label='95% (nominal gate level)')
ax.axvline(1.2, color='k', ls='--', lw=1.2, alpha=0.7, label='Paper op. point')
ax.fill_between(A_grid, 92., 96., alpha=0.12, color='#2ca02c',
                label='Paper 92–96% range')
ax.set(xlabel='Perturbation Amplitude A [m]', ylabel='NIS gate compliance [%]',
       title='Gate Compliance vs Amplitude', ylim=(60, 102))
ax.legend(fontsize=8)

fig.suptitle('Section D — Continuous CEP-vs-Amplitude Curves with Mission Profiles',
             fontsize=11, fontweight='bold')
fig.tight_layout()
plt.show()

# SMK crossing amplitudes
print('\nSMK threshold crossing amplitudes:')
for rl, col, ls, lbl in mission_profiles:
    cross = A_grid[cep_mean >= rl]
    if len(cross):
        print(f'  {lbl:<28}: SMK onset at A ≈ {cross[0]:.2f} m')
    else:
        print(f'  {lbl:<28}: no SMK reached in sweep range')

---
## Section E — NIS Compliance Surface: (A, ω) Parameter Landscape

**Paper:** Uses a single (A=1.2m, ω=0.05 rad/s) operating point and reports
92–96% NIS compliance.

**This section:** Maps the NIS compliance surface across the 2-D space of
(perturbation amplitude A, perturbation frequency ω). This identifies:
- The gate-stealth envelope (compliance ≥ 90%)
- Whether the paper's operating point lies in the interior or near the edge
- Which (A,ω) combinations produce the most effective estimator collapse

In [ ]:
N_A   = 12
N_OMG = 12
N_MC_E = 20

A_surf   = np.linspace(0.2, 2.4, N_A)
Omg_surf = np.linspace(0.01, 0.15, N_OMG)

nis_surf  = np.empty((N_A, N_OMG))
cep_surf  = np.empty((N_A, N_OMG))

print(f'Sweeping ({N_A}×{N_OMG}) grid × {N_MC_E} MC runs ...')

orig_A, orig_O = ECT.A_PERT, ECT.OMEGA
for ia, A in enumerate(tqdm(A_surf)):
    for io, omg in enumerate(Omg_surf):
        ECT.A_PERT = A
        ECT.OMEGA  = omg
        r_tmp = ECT.run_mc(n_mc=N_MC_E, verbose=False)
        _, pc = ECT.nis_compliance(r_tmp)
        nis_surf[ia, io] = pc
        _, pert_ss = ECT.cep_steady(r_tmp)
        cep_surf[ia, io] = pert_ss

ECT.A_PERT, ECT.OMEGA = orig_A, orig_O
print('Surface sweep complete.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── NIS surface ───────────────────────────────────────────────────────────────
ax = axes[0]
im = ax.contourf(Omg_surf, A_surf, nis_surf * 100,
                 levels=20, cmap='RdYlGn')
ax.contour(Omg_surf, A_surf, nis_surf * 100,
           levels=[90.], colors='k', linewidths=1.5)
ax.scatter([0.05], [1.2], color='k', s=120, zorder=5,
           marker='*', label='Paper operating point')
ax.text(0.05, 1.3, 'Paper\nop. pt.', fontsize=8, ha='center')
cb = fig.colorbar(im, ax=ax)
cb.set_label('NIS compliance [%]')
ax.set(xlabel='ω [rad/s]', ylabel='A [m]', title='NIS Gate Compliance Surface\n(black contour = 90% boundary)')
ax.legend(fontsize=8)

# ── CEP surface ───────────────────────────────────────────────────────────────
ax = axes[1]
im2 = ax.contourf(Omg_surf, A_surf, cep_surf,
                  levels=20, cmap='YlOrRd')
# R_L contours
cs = ax.contour(Omg_surf, A_surf, cep_surf,
                levels=[ECT.R_L], colors='k', linewidths=1.5)
ax.clabel(cs, fmt=f'R_L={ECT.R_L:.0f}m', fontsize=8)
ax.scatter([0.05], [1.2], color='k', s=120, zorder=5,
           marker='*', label='Paper operating point')
cb2 = fig.colorbar(im2, ax=ax)
cb2.set_label('Perturbed CEP [m]')
ax.set(xlabel='ω [rad/s]', ylabel='A [m]', title=f'Perturbed CEP Surface\n(black = R_L={ECT.R_L:.0f}m boundary)')
ax.legend(fontsize=8)

fig.suptitle('Section E — NIS Compliance & CEP Surfaces over (A, ω) Parameter Space',
             fontsize=11, fontweight='bold')
fig.tight_layout()
plt.show()

# Identify stealth-effective region
stealth = (nis_surf >= 0.90) & (cep_surf >= ECT.R_L / 2.)
print(f'\nStealth-effective region (NIS≥90% AND CEP≥R_L/2):')
print(f'  {stealth.sum()} of {N_A*N_OMG} grid points ({stealth.mean()*100:.0f}%)')

---
## Section F — Information-to-Energy Yield η_info

**Paper (Section II-D, eq. 10):**
$$\eta_{\text{info}} = \frac{\Delta h(X)}{E_{\text{attack}}}$$

where $\Delta h(X) = h_{\text{pert}} - h_{\text{nom}}$ is the differential Shannon entropy
increase in the impact-point distribution, and $E_{\text{attack}}$ is delivered energy.

**Paper example:** CEP 200m → 5km gives Δh ≈ 9.3 bits at E_attack ≈ 60 kJ → η ≈ 1.55×10⁻⁴ bits/J.

**This section:** Derives η_info continuously across CEP ratios and energy budgets,
mapping the full η_info landscape for the ECT simulation operating point.

In [ ]:
# ── Differential entropy of bivariate normal: h(X) = ln(2πe σ²) for 2D isotropic ─
# For 2D bivariate normal with covariance diag(σx², σy²):
# h(X) = ln(2πe) + 0.5·ln(σx²·σy²)  [nats] → divide by ln(2) for bits

def shannon_entropy_bits_2d(sigma_x, sigma_y):
    """Differential Shannon entropy of 2D Gaussian impact distribution [bits]."""
    return (np.log(2*np.pi*np.e) + 0.5*np.log(sigma_x**2 * sigma_y**2)) / np.log(2)

# CEP → σ inversion (isotropic): CEP = 1.1774 σ → σ = CEP / 1.1774
def cep_to_sigma(cep):
    return cep / ECT.CEP_K

# ── η_info landscape ──────────────────────────────────────────────────────────
# Axis 1: CEP ratio (perturbed / nominal) — drives Δh
# Axis 2: E_attack in kJ — denominator of η

sigma_nom = cep_to_sigma(ECT.cep_steady(res)[0])  # nominal σ from our MC

cep_ratio_grid = np.linspace(1.0, 10.0, 80)       # CEP_pert / CEP_nom
E_grid_kJ      = np.array([10., 30., 60., 120., 300.])  # energy budget [kJ]

h_nom    = shannon_entropy_bits_2d(sigma_nom, sigma_nom)

eta_matrix = np.empty((len(E_grid_kJ), len(cep_ratio_grid)))
delta_h    = np.empty(len(cep_ratio_grid))

for j, ratio in enumerate(cep_ratio_grid):
    sigma_p  = cep_to_sigma(sigma_nom * ECT.CEP_K * ratio)  # perturbed σ
    h_pert   = shannon_entropy_bits_2d(sigma_p, sigma_p)
    delta_h[j] = h_pert - h_nom
    for i, E_kJ in enumerate(E_grid_kJ):
        eta_matrix[i, j] = delta_h[j] / (E_kJ * 1e3)  # bits/J

# Paper operating point
nom_ss, pert_ss = ECT.cep_steady(res)
op_ratio = pert_ss / nom_ss
print(f'Paper sim operating point: CEP_nom={nom_ss:.2f}m, CEP_pert={pert_ss:.2f}m, ratio={op_ratio:.2f}')
print(f'Paper example (200m→5km): ratio={5000/200:.0f}x, Δh≈9.3 bits at 60kJ → η={9.3/60e3:.2e} bits/J')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Δh vs CEP ratio ───────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(cep_ratio_grid, delta_h, color='#1f77b4', lw=2.0)
ax.axvline(op_ratio, color='#ff7f0e', ls='--', lw=1.5,
           label=f'Sim op. point (ratio={op_ratio:.2f}x)')
ax.axvline(5000/200., color='k', ls=':', lw=1.4,
           label='Paper example (200m→5km, 25x)')

# Mark paper's 9.3 bits point
ax.axhline(9.3, color='k', ls=':', lw=1.0, alpha=0.5)
ax.text(9.0, 9.5, 'Paper: 9.3 bits', fontsize=8)

ax.set(xlabel='CEP ratio (pert / nom)', ylabel='Δh(X) [bits]',
       title='Shannon Entropy Increase vs CEP Degradation')
ax.legend(fontsize=8)

# ── η_info vs CEP ratio for multiple energy budgets ───────────────────────────
ax = axes[1]
colors_e = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']
for i, (E_kJ, col) in enumerate(zip(E_grid_kJ, colors_e)):
    ax.semilogy(cep_ratio_grid, eta_matrix[i], color=col, lw=1.7,
                label=f'E={E_kJ:.0f} kJ')

ax.axvline(op_ratio, color='#ff7f0e', ls='--', lw=1.4,
           label=f'Sim op. point ({op_ratio:.2f}x)')

# Paper example marker
paper_eta = 9.3 / 60e3
ax.scatter([25.], [paper_eta], s=80, color='k', zorder=5, marker='*',
           label=f'Paper example ({paper_eta:.1e} bits/J)')

ax.set(xlabel='CEP ratio (pert / nom)', ylabel='η_info [bits/J]',
       title='Information-to-Energy Yield η_info')
ax.legend(fontsize=8, loc='lower right')

fig.suptitle('Section F — η_info Characterisation (eq. 10)', fontsize=11, fontweight='bold')
fig.tight_layout()
plt.show()

# Operating point summary
op_dh  = delta_h[np.argmin(np.abs(cep_ratio_grid - op_ratio))]
print(f'\nη_info at simulation operating point (ratio={op_ratio:.2f}x):')
for E_kJ in E_grid_kJ:
    eta_op = op_dh / (E_kJ * 1e3)
    print(f'  E={E_kJ:>5.0f} kJ → η_info = {eta_op:.3e} bits/J  (Δh={op_dh:.2f} bits)')

---
## Section G — Consolidated Summary Table

Comparison of paper point-estimates vs this notebook's 95% CI equivalents.

In [ ]:
nom_ss_f, pert_ss_f = ECT.cep_steady(res)
nc_f, pc_f = ECT.nis_compliance(res)
gamma_tf, gamma_allf = ECT.gamma_series(res)
frac_f = (gamma_allf[:,-1] >= ECT.GAMMA_CRIT).mean()*100
mki_f  = pert_ss_f / ECT.R_L

# Section B CI results
c_lo_b, c_hi_b = np.percentile(cep_perturbed, [2.5, 97.5])
n_lo_b, n_hi_b = np.percentile(nis_compliance*100, [2.5, 97.5])
f_lo_b, f_hi_b = np.percentile(frac_exceed*100, [2.5, 97.5])

rows = [
    ('Metric', 'Paper (point est.)', f'This NB (N={N_SUPP})', '95% CI (Sec B)'),
    ('-'*30, '-'*20, '-'*18, '-'*22),
    ('Nominal CEP [m]', '3.2', f'{nom_ss_f:.2f}', '—'),
    ('Perturbed CEP [m]', '7.9', f'{pert_ss_f:.2f}', f'[{c_lo_b:.1f}, {c_hi_b:.1f}]'),
    ('CEP degradation', '+147%', f'+{100*(pert_ss_f/nom_ss_f-1):.0f}%', '—'),
    ('Runs > Γ_crit [%]', '100%', f'{frac_f:.0f}%', f'[{f_lo_b:.0f}%, {f_hi_b:.0f}%]'),
    ('NIS compliance [%]', '92–96%', f'{pc_f*100:.1f}%', f'[{n_lo_b:.0f}%, {n_hi_b:.0f}%]'),
    ('MKI (R_L=15m)', '0.53', f'{mki_f:.3f}', '—'),
    ('Γ_crit', '6.5 (fixed)', '6.5 (fixed)', '—'),
]

print('\nSummary: Paper vs Supplementary Notebook')
print('='*75)
for row in rows:
    print(f'{row[0]:<30} {row[1]:<22} {row[2]:<20} {row[3]}')
print('='*75)

print(f'''
CRediT footprint added by this notebook:
  ✓ Formal Analysis  — Section B quantifies Γ_crit boundary as distributional result
  ✓ Methodology      — Section C provides first quantitative V_i sensitivity ranking
  ✓ Validation       — Section D maps the full gate-stealth operating envelope
  ✓ Visualisation    — Sections E/F add parameter-space surfaces and η_info curves
  ✓ Software         — Reproducible notebook archived alongside simulation engine
''')

---
### Reproduction Notes

- **For publication-quality runs:** set `N_SUPP=500`, `N_SWEEP=100`, `N_MC_B=50` and re-run all cells.
- **Dependencies:** `numpy`, `scipy`, `matplotlib`, `tqdm` — all in `requirements.txt`.
- **Kernel:** Python 3.8+. Activate the repo venv: `source .venv/bin/activate`.
- **Core module:** `ECT_3D_Simulation_v141.py` must be in the same directory.

```bibtex
@software{barua_ect_2026,
  author  = {Barua, Nick and Douglas, R. J.},
  title   = {{Estimator Collapse Theory (ECT) Framework}},
  year    = {2026},
  doi     = {10.5281/zenodo.20037820}
}
```